# Simulator Sistem Pendingin Kapal

Simulator sistem pendingin kapal (air laut, LT, dan HT) dengan skenario operasional dan misi. Ini versi Python dari aplikasi web, jadi bisa dijalankan di Google Colab tanpa instalasi apa pun.

**Cara pakai**
1. Jalankan sel dari atas ke bawah (menu *Runtime > Run all*).
2. Bagian *Panel interaktif* di paling bawah berisi tombol dan slider untuk mengendalikan kapal.
3. Mau melihat kodenya? Sel 1 sampai 4 berisi mesin simulasinya.

Jangan lupa: grafik dan panel baru bergerak setelah tombol Play di panel ditekan.

## 1. Mesin fisika

In [ ]:
"""
Mesin fisika simulator sistem pendingin kapal.

Ini terjemahan dari src/engine/physics.ts pada versi web (React).
Setiap kali fungsi langkah_simulasi() dipanggil, kondisi kapal maju sebesar dt detik.
"""

import math


def batasi(nilai, kecil, besar):
    """Jaga supaya nilai tidak keluar dari rentang kecil..besar."""
    return max(kecil, min(besar, nilai))


def mendekati(sekarang, target, laju):
    """Geser nilai sekarang menuju target. Makin besar laju, makin cepat."""
    return sekarang + (target - sekarang) * batasi(laju, 0, 1)


def bulatkan(x):
    """Pembulatan biasa (0,5 naik). round() bawaan Python membulatkan ke genap."""
    return int(math.floor(x + 0.5))


def tambah_alarm_jika_baru(alarms, waktu, tingkat, sumber, pesan):
    """
    Tambah alarm ke daftar (yang terbaru ditaruh paling depan).
    Alarm dengan sumber dan tingkat yang sama diabaikan kalau baru muncul < 20 detik lalu.
    """
    for a in alarms:
        if a["sumber"] == sumber and a["tingkat"] == tingkat and waktu - a["waktu"] < 20:
            return alarms

    baru = {
        "waktu": waktu,
        "tingkat": tingkat,  # 'info', 'warning', atau 'critical'
        "sumber": sumber,
        "pesan": pesan,
        "acknowledged": False,
    }
    return ([baru] + alarms)[:40]


def faktor_katup(pompa):
    """Seberapa besar aliran yang lewat pompa ini (0 sampai 1). 0 kalau mati/trip/katup tutup."""
    if pompa["running"] and not pompa["tripped"] and pompa["valve_open"] and pompa["valve_pct"] > 0:
        return (pompa["valve_pct"] / 100) ** 0.78
    return 0.0


def motor_hidup(pompa):
    return pompa["running"] and not pompa["tripped"]


def gabung_pompa(f1, f2, faktor_kedua):
    """
    Dua pompa paralel tidak menghasilkan aliran dua kali lipat.
    Kelebihan di atas 1.0 hanya dihitung sebagian (faktor_kedua).
    """
    total = f1 + f2
    if total > 1.0:
        return 1.0 + (total - 1.0) * faktor_kedua
    return total


# Batas jumlah titik riwayat yang disimpan (cukup untuk sekitar 25 menit waktu simulasi)
MAKS_RIWAYAT = 2000


def langkah_simulasi(s, dt_detik):
    """
    Majukan simulasi sebesar dt_detik (dikali kecepatan simulasi).
    s adalah dictionary kondisi kapal dari scenarios.py. Isinya diubah langsung (tidak dikembalikan salinan).
    """
    if not s["running"]:
        return s

    dt = dt_detik * s["speed_multiplier"]
    waktu = s["elapsed_seconds"] + dt

    p = s["pumps"]

    # 1. Beban dan putaran mesin utama
    target_load = 0 if s["engine_tripped"] else s["telegraph_pct"]
    actual_load_pct = mendekati(s["actual_load_pct"], target_load, dt * 0.45)
    engine_rpm = bulatkan(22 + (actual_load_pct / 100) * 98) if actual_load_pct > 1 else 0

    # 2. Hidrolik air laut (sea chest, overboard, katup tiap pompa)
    sw1_active = motor_hidup(p["sw1"]) and p["sw1"]["valve_open"] and p["sw1"]["valve_pct"] > 0
    sw2_active = motor_hidup(p["sw2"]) and p["sw2"]["valve_open"] and p["sw2"]["valve_pct"] > 0

    sw1_valve_factor = faktor_katup(p["sw1"])
    sw2_valve_factor = faktor_katup(p["sw2"])
    sw_combined = gabung_pompa(sw1_valve_factor, sw2_valve_factor, 0.62)

    sea_chest_valve_factor = (s["sea_chest_valve_pct"] / 100) ** 0.75 if s["sea_chest_valve_open"] else 0
    overboard_valve_factor = (s["overboard_valve_pct"] / 100) ** 0.75 if s["overboard_valve_open"] else 0
    sw_path_factor = min(sea_chest_valve_factor, overboard_valve_factor)

    strainer_factor = batasi(1 - (s["strainer_fouling"] / 100) * 0.72, 0.18, 1)
    sea_chest_factor = 0.96 if s["sea_chest_mode"] == "high" else 1.0
    sw_target_flow = 420 * sw_combined * sw_path_factor * strainer_factor * sea_chest_factor
    sw_flow_total = mendekati(s["sw_flow_total"], sw_target_flow, dt * 1.6)

    sw_motor_count = (1 if motor_hidup(p["sw1"]) else 0) + (1 if motor_hidup(p["sw2"]) else 0)
    if sw_motor_count == 0:
        sw_target_pressure = 0.05
    elif (not s["sea_chest_valve_open"]) or s["sea_chest_valve_pct"] == 0:
        sw_target_pressure = 0.15  # hisap kering / kavitasi
    elif s["overboard_valve_open"] and s["overboard_valve_pct"] > 5 and (sw1_active or sw2_active):
        sw_target_pressure = (1.7 + sw_combined * 0.65) * (0.45 + 0.55 * strainer_factor)
    else:
        sw_target_pressure = 3.6  # tekanan buntu (deadhead), discharge atau overboard tertutup
    sw_pressure = mendekati(s["sw_pressure"], sw_target_pressure, dt * 1.8)
    if sw_flow_total > 10:
        strainer_delta_p = batasi((s["strainer_fouling"] / 100) * 1.45 * (sw_flow_total / 320), 0.05, 1.8)
    else:
        strainer_delta_p = 0

    # 3. Hidrolik air tawar LT
    lt1_factor = faktor_katup(p["lt1"])
    lt2_factor = faktor_katup(p["lt2"])
    lt_combined = gabung_pompa(lt1_factor, lt2_factor, 0.56)
    lt_active_count = (1 if lt1_factor > 0 else 0) + (1 if lt2_factor > 0 else 0)
    lt_target_flow = 360 * lt_combined
    lt_flow_total = mendekati(s["lt_flow_total"], lt_target_flow, dt * 1.8)
    if motor_hidup(p["lt1"]) or motor_hidup(p["lt2"]):
        lt_target_pressure = 3.4 if lt_active_count == 0 else 2.3 + lt_combined * 0.5
    else:
        lt_target_pressure = 0.2
    lt_pressure = mendekati(s["lt_pressure"], lt_target_pressure, dt * 1.8)

    # 4. Hidrolik air tawar HT
    ht1_factor = faktor_katup(p["ht1"])
    ht2_factor = faktor_katup(p["ht2"])
    ht_combined = gabung_pompa(ht1_factor, ht2_factor, 0.54)
    ht_active_count = (1 if ht1_factor > 0 else 0) + (1 if ht2_factor > 0 else 0)
    ht_target_flow = 240 * ht_combined
    ht_flow_total = mendekati(s["ht_flow_total"], ht_target_flow, dt * 1.8)
    if motor_hidup(p["ht1"]) or motor_hidup(p["ht2"]):
        ht_target_pressure = 3.6 if ht_active_count == 0 else 2.5 + ht_combined * 0.6
    else:
        ht_target_pressure = 0.4
    ht_pressure = mendekati(s["ht_pressure"], ht_target_pressure, dt * 1.8)

    # Isi aliran dan tekanan tiap pompa (tampilan)
    def perbarui_pompa(id_pompa, faktor, faktor_total, aliran_total, tekanan_sistem):
        pp = p[id_pompa]
        hidup = motor_hidup(pp)
        pp["flow_rate"] = (aliran_total * faktor) / faktor_total if (hidup and faktor_total > 0.01) else 0
        pp["pressure"] = tekanan_sistem if hidup else 0

    perbarui_pompa("sw1", sw1_valve_factor, sw1_valve_factor + sw2_valve_factor, sw_flow_total, sw_pressure)
    perbarui_pompa("sw2", sw2_valve_factor, sw1_valve_factor + sw2_valve_factor, sw_flow_total, sw_pressure)
    perbarui_pompa("lt1", lt1_factor, lt1_factor + lt2_factor, lt_flow_total, lt_pressure)
    perbarui_pompa("lt2", lt2_factor, lt1_factor + lt2_factor, lt_flow_total, lt_pressure)
    perbarui_pompa("ht1", ht1_factor, ht1_factor + ht2_factor, ht_flow_total, ht_pressure)
    perbarui_pompa("ht2", ht2_factor, ht1_factor + ht2_factor, ht_flow_total, ht_pressure)

    # 5. Filter pelan-pelan kotor kalau air laut mengalir (kecuali skenario warmup)
    strainer_fouling = s["strainer_fouling"]
    if sw_flow_total > 50 and s["scenario_id"] != "warmup":
        strainer_fouling = batasi(strainer_fouling + dt * 0.015, 0, 95)

    # 6. Pengatur katup 3-arah
    lt3way_cooler_pct = s["lt3way_cooler_pct"]
    if s["lt3way_mode"] == "auto":
        lt_error = s["lt_supply_temp"] - 36.0
        target_pct = batasi(50 + lt_error * 9.5, 5, 100)
        lt3way_cooler_pct = mendekati(lt3way_cooler_pct, target_pct, dt * 0.65)

    ht3way_cooler_pct = s["ht3way_cooler_pct"]
    if s["ht3way_mode"] == "auto":
        ht_error = s["ht_outlet_temp"] - 82.0
        target_pct = batasi(45 + ht_error * 11, 0, 100)
        ht3way_cooler_pct = mendekati(ht3way_cooler_pct, target_pct, dt * 0.7)

    # 7. Neraca energi (panas)
    load_factor = (actual_load_pct / 100) ** 1.25
    aux_load_factor = (s["aux_load_pct"] / 100) ** 1.18
    suhu_ruang = 31

    # Panas masuk dari mesin utama dan auxiliary engine
    jacket_heat_input = 4.0 + load_factor * 105.0 + (24.0 if s["jacket_heater_on"] else 0)
    lube_oil_heat_input = 2.0 + load_factor * 44.0
    scavenge_heat_input = 1.5 + load_factor * 52.0
    aux_heat_input = 2.5 + aux_load_factor * 36.0

    # Pemindahan panas HT -> LT
    ht_cooling_flow_factor = batasi(ht_flow_total / 240, 0.05, 1.35) * (ht3way_cooler_pct / 100)
    lt_ht_exchanger_factor = batasi(lt_flow_total / 360, 0.04, 1.3)
    ht_to_lt_coeff = min(ht_cooling_flow_factor, lt_ht_exchanger_factor) * 1.75
    ht_delta_t = max(0, s["ht_outlet_temp"] - s["lt_return_temp"])
    ht_heat_rejected = ht_delta_t * ht_to_lt_coeff

    ht_thermal_mass = 28.0 if ht_flow_total > 40 else 9.5
    ht_ambient_loss = (s["ht_outlet_temp"] - suhu_ruang) * 0.08
    d_ht_outlet = ((jacket_heat_input - ht_heat_rejected - ht_ambient_loss) / ht_thermal_mass) * dt
    ht_outlet_temp = batasi(s["ht_outlet_temp"] + d_ht_outlet, s["sea_water_temp"], 118)

    if ht_flow_total > 30:
        ht_inlet_target = batasi(
            ht_outlet_temp - (jacket_heat_input / max(80, ht_flow_total)) * 16,
            s["lt_supply_temp"],
            ht_outlet_temp,
        )
    else:
        ht_inlet_target = ht_outlet_temp - 1.5
    ht_inlet_temp = mendekati(s["ht_inlet_temp"], ht_inlet_target, dt * 0.85)

    # Lube oil, scavenge air, dan auxiliary engine didinginkan oleh LT
    lt_cooling_flow_norm = batasi(lt_flow_total / 360, 0.03, 1.35)
    lo_cooling = max(0, s["lube_oil_temp"] - s["lt_supply_temp"]) * lt_cooling_flow_norm * 2.1
    lo_ambient = (s["lube_oil_temp"] - suhu_ruang) * 0.05
    lube_oil_temp = batasi(
        s["lube_oil_temp"] + ((lube_oil_heat_input - lo_cooling - lo_ambient) / 18) * dt, 24, 105
    )

    scav_cooling = max(0, s["scavenge_air_temp"] - s["lt_supply_temp"]) * lt_cooling_flow_norm * 2.4
    scav_ambient = (s["scavenge_air_temp"] - suhu_ruang) * 0.06
    scavenge_air_temp = batasi(
        s["scavenge_air_temp"] + ((scavenge_heat_input - scav_cooling - scav_ambient) / 14) * dt, 24, 105
    )

    aux_valve_factor = (s["aux_valve_pct"] / 100) ** 0.75 if s["aux_valve_open"] else 0
    aux_cooling = (
        max(0, s["aux_engine_temp"] - s["lt_supply_temp"]) * lt_cooling_flow_norm * aux_valve_factor * 2.25
    )
    aux_ambient = (s["aux_engine_temp"] - suhu_ruang) * 0.06
    aux_engine_temp = batasi(
        s["aux_engine_temp"] + ((aux_heat_input - aux_cooling - aux_ambient) / 15) * dt, 24, 105
    )

    # Suhu LT return mengumpulkan panas dari lube oil, scavenge air, aux engine, dan penukar HT/LT
    total_lt_heat_load = ht_heat_rejected + lo_cooling * 0.65 + scav_cooling * 0.65 + aux_cooling * 0.7
    if lt_flow_total > 30:
        lt_return_target = batasi(
            s["lt_supply_temp"] + (total_lt_heat_load / max(120, lt_flow_total)) * 28,
            s["lt_supply_temp"],
            96,
        )
    else:
        lt_return_target = s["lt_return_temp"] + (total_lt_heat_load / 40) * dt
    lt_return_temp = mendekati(s["lt_return_temp"], lt_return_target, dt * 0.9)

    # Central cooler: LT <-> air laut
    cooler_efficiency = batasi(1 - (s["cooler_fouling"] / 100) * 0.65, 0.25, 1.0)
    sw_effectiveness = batasi(sw_flow_total / 380, 0, 1.35) * cooler_efficiency
    lt_through_cooler = (lt3way_cooler_pct / 100) * lt_cooling_flow_norm
    central_coeff = min(sw_effectiveness, lt_through_cooler) * 2.75

    central_delta_t = max(0, lt_return_temp - s["sea_water_temp"])
    heat_rejected_to_sea = central_delta_t * central_coeff

    lt_thermal_mass = 22.0 if lt_flow_total > 40 else 12.0
    lt_net_heat = total_lt_heat_load - heat_rejected_to_sea - (s["lt_supply_temp"] - suhu_ruang) * 0.06
    lt_supply_temp = batasi(
        s["lt_supply_temp"] + (lt_net_heat / lt_thermal_mass) * dt, s["sea_water_temp"], 92
    )

    # Suhu air laut
    sw_supply_temp = s["sea_water_temp"]
    if sw_flow_total > 20:
        sw_discharge_target = batasi(
            sw_supply_temp + (heat_rejected_to_sea / max(100, sw_flow_total)) * 34, sw_supply_temp, 68
        )
    else:
        sw_discharge_target = sw_supply_temp
    sw_discharge_temp = mendekati(s["sw_discharge_temp"], sw_discharge_target, dt * 0.9)

    # 8. Alarm dan trip pengaman mesin
    alarms = s["alarms"]
    engine_tripped = s["engine_tripped"]
    engine_trip_reason = s["engine_trip_reason"]

    if actual_load_pct > 40 and ht_outlet_temp < 50:
        alarms = tambah_alarm_jika_baru(
            alarms, waktu, "warning", "Main Engine",
            f"Beban tinggi ({actual_load_pct:.0f}%) saat mesin masih dingin ({ht_outlet_temp:.1f}°C)! Risiko thermal shock.",
        )
    if aux_engine_temp >= 78:
        alarms = tambah_alarm_jika_baru(
            alarms, waktu, "warning", "Auxiliary Engine",
            f"Suhu Auxiliary Engine tinggi ({aux_engine_temp:.1f}°C)! Periksa katup pendingin Aux ({s['aux_valve_pct']:.0f}%) atau aliran LT.",
        )
    if 88 <= ht_outlet_temp < 95:
        alarms = tambah_alarm_jika_baru(
            alarms, waktu, "warning", "HT Jacket Water",
            f"Suhu air jaket tinggi ({ht_outlet_temp:.1f}°C). Periksa aliran SW, fouling cooler, atau turunkan beban.",
        )
    if ht_outlet_temp >= 95 and not engine_tripped:
        alarms = tambah_alarm_jika_baru(
            alarms, waktu, "critical", "HT Jacket Water",
            f"KRITIS: Suhu jaket mencapai {ht_outlet_temp:.1f}°C! Auto-shutdown aktif pada 98°C.",
        )
    if ht_outlet_temp >= 98 and not engine_tripped:
        engine_tripped = True
        engine_trip_reason = "Overheat Air Jaket HT (≥ 98°C)"
        alarms = tambah_alarm_jika_baru(
            alarms, waktu, "critical", "Safety Shutdown",
            "AUTO-TRIP MESIN UTAMA: Suhu HT mencapai 98°C untuk mencegah kerusakan silinder!",
        )
    if lube_oil_temp >= 78 and not engine_tripped:
        engine_tripped = True
        engine_trip_reason = "Overheat Oli Pelumas (≥ 78°C)"
        alarms = tambah_alarm_jika_baru(
            alarms, waktu, "critical", "Safety Shutdown",
            "AUTO-TRIP MESIN UTAMA: Suhu Lube Oil melebihi batas aman (78°C)!",
        )
    if strainer_delta_p >= 0.95:
        alarms = tambah_alarm_jika_baru(
            alarms, waktu, "warning", "SW Strainer",
            f"Filter air laut tersumbat (ΔP {strainer_delta_p:.2f} bar). Lakukan backwash.",
        )
    if (not s["overboard_valve_open"] or s["overboard_valve_pct"] < 10) and sw_motor_count > 0:
        alarms = tambah_alarm_jika_baru(
            alarms, waktu, "warning", "Overboard Valve",
            "Pompa SW menyala tetapi katup Overboard tertutup/tercekik! Aliran pendinginan terhenti.",
        )
    if (not s["sea_chest_valve_open"] or s["sea_chest_valve_pct"] < 10) and sw_motor_count > 0:
        alarms = tambah_alarm_jika_baru(
            alarms, waktu, "warning", "Sea Chest Valve",
            "Katup hisap Sea Chest tertutup saat pompa SW berjalan! Risiko kavitasi pompa.",
        )

    # 9. Penghitung waktu misi dan pengecekan selesai
    mission_hold = s["mission_hold_seconds"]
    mission_completed = s["mission_completed"]
    mission_failed = s["mission_failed"]

    if s["scenario_id"] == "warmup":
        ok = (
            actual_load_pct >= 70
            and 74 <= ht_outlet_temp <= 87
            and s["overboard_valve_open"]
            and sw_flow_total > 250
            and not engine_tripped
        )
        mission_hold = mission_hold + dt if ok else max(0, mission_hold - dt * 0.5)
        if mission_hold >= 12:
            mission_completed = True
    elif s["scenario_id"] == "tropical":
        ok = (
            s["strainer_fouling"] <= 20
            and s["cooler_fouling"] <= 20
            and p["sw1"]["running"]
            and p["sw2"]["running"]
            and actual_load_pct >= 80
            and ht_outlet_temp < 87
            and not engine_tripped
        )
        mission_hold = mission_hold + dt if ok else max(0, mission_hold - dt * 0.5)
        if mission_hold >= 15:
            mission_completed = True
    elif s["scenario_id"] == "emergency":
        if engine_tripped:
            mission_failed = True
        ok = (
            p["sw2"]["running"]
            and sw_flow_total > 250
            and (s["lt3way_mode"] == "auto" or s["lt3way_cooler_pct"] >= 70)
            and actual_load_pct >= 75
            and ht_outlet_temp < 87
            and not engine_tripped
        )
        mission_hold = mission_hold + dt if ok else max(0, mission_hold - dt * 0.5)
        if mission_hold >= 15 and not mission_failed:
            mission_completed = True

    # 10. Catat riwayat (sekitar tiap 0,8 detik waktu simulasi)
    riwayat = s["history"]
    waktu_terakhir = riwayat[-1]["time"] if riwayat else -1
    if waktu - waktu_terakhir >= 0.8:
        riwayat.append({
            "time": round(waktu, 1),
            "ht_outlet": round(ht_outlet_temp, 2),
            "lt_supply": round(lt_supply_temp, 2),
            "lube_oil": round(lube_oil_temp, 2),
            "scavenge_air": round(scavenge_air_temp, 2),
            "aux_engine": round(aux_engine_temp, 2),
            "engine_load": round(actual_load_pct, 1),
            "sw_flow": round(sw_flow_total),
        })
        if len(riwayat) > MAKS_RIWAYAT:
            del riwayat[0]

    # Simpan hasil perhitungan ke kondisi kapal
    s.update({
        "elapsed_seconds": waktu,
        "actual_load_pct": actual_load_pct,
        "engine_rpm": engine_rpm,
        "engine_tripped": engine_tripped,
        "engine_trip_reason": engine_trip_reason,
        "aux_engine_temp": aux_engine_temp,
        "lt3way_cooler_pct": lt3way_cooler_pct,
        "ht3way_cooler_pct": ht3way_cooler_pct,
        "strainer_fouling": strainer_fouling,
        "sw_supply_temp": sw_supply_temp,
        "sw_discharge_temp": sw_discharge_temp,
        "sw_flow_total": sw_flow_total,
        "sw_pressure": sw_pressure,
        "strainer_delta_p": strainer_delta_p,
        "lt_supply_temp": lt_supply_temp,
        "lt_return_temp": lt_return_temp,
        "lt_flow_total": lt_flow_total,
        "lt_pressure": lt_pressure,
        "ht_inlet_temp": ht_inlet_temp,
        "ht_outlet_temp": ht_outlet_temp,
        "ht_flow_total": ht_flow_total,
        "ht_pressure": ht_pressure,
        "lube_oil_temp": lube_oil_temp,
        "scavenge_air_temp": scavenge_air_temp,
        "mission_hold_seconds": mission_hold,
        "mission_completed": mission_completed,
        "mission_failed": mission_failed,
        "alarms": alarms,
    })
    return s

## 2. Skenario dan misi

In [ ]:
"""
Skenario dan misi simulator.
Terjemahan dari src/engine/scenarios.ts pada versi web.

Kondisi kapal disimpan dalam satu dictionary (sebut saja s). Contoh cara pakai:
    s = SCENARIOS["tropical"]["buat_kondisi"]()
    print(s["ht_outlet_temp"])
"""


def buat_pompa(**override):
    """
    Buat 6 pompa (SW, LT, HT masing-masing 2 unit).
    Contoh: buat_pompa(sw1={"running": False}) untuk mematikan pompa SW #1.
    """
    daftar = [
        # id, nama, sirkuit, nyala dari awal?, kapasitas (m3/h)
        ("sw1", "Pompa Air Laut #1 (Utama)", "sw", True, 420),
        ("sw2", "Pompa Air Laut #2 (Cadangan)", "sw", False, 420),
        ("lt1", "Pompa Air Tawar LT #1", "lt", True, 360),
        ("lt2", "Pompa Air Tawar LT #2", "lt", False, 360),
        ("ht1", "Pompa Air Tawar HT #1", "ht", True, 240),
        ("ht2", "Pompa Air Tawar HT #2", "ht", False, 240),
    ]
    pompa = {}
    for id_pompa, nama, sirkuit, nyala, kapasitas in daftar:
        ubah = override.get(id_pompa, {})
        pompa[id_pompa] = {
            "id": id_pompa,
            "name": nama,
            "circuit": sirkuit,
            "running": ubah.get("running", nyala),
            "tripped": ubah.get("tripped", False),
            "valve_open": ubah.get("valve_open", True),
            "valve_pct": ubah.get("valve_pct", 100),
            "flow_rate": 0,
            "rated_flow": kapasitas,
            "pressure": 0,
        }
    return pompa


def alarm_awal(tingkat, sumber, pesan, acknowledged=False):
    return {"waktu": 0, "tingkat": tingkat, "sumber": sumber, "pesan": pesan, "acknowledged": acknowledged}


def kondisi_dasar(scenario_id, **isi):
    """Nilai-nilai yang sama di semua skenario. Sisanya diisi/ditimpa lewat **isi."""
    s = {
        "running": True,
        "speed_multiplier": 1,
        "elapsed_seconds": 0,
        "scenario_id": scenario_id,
        "sea_chest_mode": "low",
        "sea_chest_valve_open": True,
        "sea_chest_valve_pct": 100,
        "overboard_valve_open": True,
        "overboard_valve_pct": 100,
        "engine_tripped": False,
        "engine_trip_reason": None,
        "aux_valve_open": True,
        "aux_valve_pct": 100,
        "lt3way_mode": "auto",
        "ht3way_mode": "auto",
        "jacket_heater_on": False,
        "mission_hold_seconds": 0,
        "mission_completed": False,
        "mission_failed": False,
        "history": [],
    }
    s.update(isi)
    return s


# ---------------------------------------------------------------- Mode bebas

def buat_sandbox():
    return kondisi_dasar(
        "sandbox",
        sea_water_temp=24,
        telegraph_pct=65, actual_load_pct=65, engine_rpm=82,
        aux_load_pct=45, aux_engine_temp=47.4,
        pumps=buat_pompa(),
        lt3way_cooler_pct=58, ht3way_cooler_pct=52,
        strainer_fouling=8, cooler_fouling=5,
        sw_supply_temp=24, sw_discharge_temp=31.4, sw_flow_total=400, sw_pressure=2.3, strainer_delta_p=0.14,
        lt_supply_temp=36.2, lt_return_temp=44.8, lt_flow_total=360, lt_pressure=2.8,
        ht_inlet_temp=74.5, ht_outlet_temp=81.8, ht_flow_total=240, ht_pressure=3.1,
        lube_oil_temp=48.2, scavenge_air_temp=42.6,
        alarms=[alarm_awal(
            "info", "Sistem ECR",
            "Simulator siap. Kondisi laut normal 24°C pada beban mesin 65% & Aux Engine 45%.", True)],
    )


# ---------------------------------------------------------------- Misi 1

def buat_warmup():
    return kondisi_dasar(
        "warmup",
        sea_water_temp=20,
        overboard_valve_open=False,
        telegraph_pct=0, actual_load_pct=0, engine_rpm=0,
        aux_load_pct=35, aux_engine_temp=42,
        pumps=buat_pompa(sw1={"running": False}, lt1={"running": False}, ht1={"running": False}),
        lt3way_cooler_pct=10, ht3way_cooler_pct=5,
        strainer_fouling=5, cooler_fouling=0,
        sw_supply_temp=20, sw_discharge_temp=20, sw_flow_total=0, sw_pressure=0, strainer_delta_p=0,
        lt_supply_temp=26, lt_return_temp=26, lt_flow_total=0, lt_pressure=0,
        ht_inlet_temp=28, ht_outlet_temp=28, ht_flow_total=0, ht_pressure=0,
        lube_oil_temp=28, scavenge_air_temp=26,
        alarms=[alarm_awal(
            "info", "Perwira Jaga",
            "Persiapan berangkat: sirkulasi air tawar & pemanasan jaket mesin diperlukan.")],
    )


def tujuan_warmup(s):
    sirkulasi = s["ht_flow_total"] > 150 and s["lt_flow_total"] > 200
    panas = s["ht_outlet_temp"] >= 58
    sw_aktif = s["overboard_valve_open"] and s["sw_flow_total"] > 250
    berlayar = (s["actual_load_pct"] >= 70 and 74 <= s["ht_outlet_temp"] <= 87 and not s["engine_tripped"])
    stabil = s["mission_hold_seconds"] >= 12
    return [
        {"id": "circ", "label": "Jalankan pompa sirkulasi HT & LT", "completed": sirkulasi,
         "detail": "Sirkulasi aktif" if sirkulasi else "Nyalakan Pompa HT #1 & LT #1"},
        {"id": "preheat", "label": "Panaskan air jaket HT ≥ 58°C", "completed": panas,
         "detail": f"{s['ht_outlet_temp']:.1f}°C / 58.0°C"},
        {"id": "sw", "label": "Buka katup Overboard & nyalakan Pompa SW", "completed": sw_aktif,
         "detail": f"Aliran SW {s['sw_flow_total']:.0f} m³/h" if sw_aktif else "Katup & pompa SW belum aktif"},
        {"id": "cruise", "label": "Berlayar ≥ 70% beban & stabil 12 detik (74–87°C)", "completed": stabil,
         "detail": (f"Stabilisasi {min(12, s['mission_hold_seconds']):.0f}/12 dtk" if berlayar
                    else f"Beban {s['actual_load_pct']:.0f}% · HT {s['ht_outlet_temp']:.1f}°C")},
    ]


# ---------------------------------------------------------------- Misi 2

def buat_tropical():
    return kondisi_dasar(
        "tropical",
        sea_water_temp=32,
        telegraph_pct=85, actual_load_pct=85, engine_rpm=104,
        aux_load_pct=65, aux_engine_temp=58.5,
        pumps=buat_pompa(),
        lt3way_cooler_pct=92, ht3way_cooler_pct=85,
        strainer_fouling=68, cooler_fouling=52,
        sw_supply_temp=32, sw_discharge_temp=44.5, sw_flow_total=195, sw_pressure=1.4, strainer_delta_p=1.15,
        lt_supply_temp=44.2, lt_return_temp=54.0, lt_flow_total=360, lt_pressure=2.7,
        ht_inlet_temp=81.0, ht_outlet_temp=89.4, ht_flow_total=240, ht_pressure=3.1,
        lube_oil_temp=61.5, scavenge_air_temp=54.2,
        alarms=[
            alarm_awal("warning", "SW Strainer",
                       "Tekanan diferensial filter air laut tinggi (1.15 bar). Segera bersihkan/backwash!"),
            alarm_awal("warning", "Central Cooler",
                       "Efisiensi penukar panas menurun akibat fouling (52%) pada air laut 32°C."),
        ],
    )


def tujuan_tropical(s):
    filter_bersih = s["strainer_fouling"] <= 20
    cooler_bersih = s["cooler_fouling"] <= 20
    dua_pompa = s["pumps"]["sw1"]["running"] and s["pumps"]["sw2"]["running"]
    stabil = s["mission_hold_seconds"] >= 15
    return [
        {"id": "strainer", "label": "Backwash Filter SW (Fouling ≤ 20%)", "completed": filter_bersih,
         "detail": f"Fouling: {s['strainer_fouling']:.0f}%"},
        {"id": "cooler", "label": "Bersihkan pelat Central Cooler (≤ 20%)", "completed": cooler_bersih,
         "detail": f"Kerak: {s['cooler_fouling']:.0f}%"},
        {"id": "parallel", "label": "Aktifkan Pompa SW #1 & #2 paralel", "completed": dua_pompa,
         "detail": f"Aliran {s['sw_flow_total']:.0f} m³/h" if dua_pompa else "Jalankan kedua pompa SW"},
        {"id": "temp", "label": "Pertahankan HT < 87°C & beban ≥ 80% selama 15 detik", "completed": stabil,
         "detail": f"{min(15, s['mission_hold_seconds']):.0f}/15 dtk · HT {s['ht_outlet_temp']:.1f}°C"},
    ]


# ---------------------------------------------------------------- Misi 3

def buat_emergency():
    return kondisi_dasar(
        "emergency",
        sea_water_temp=28,
        telegraph_pct=88, actual_load_pct=88, engine_rpm=108,
        aux_load_pct=60, aux_engine_temp=62.4,
        pumps=buat_pompa(sw1={"running": False, "tripped": True}, sw2={"running": False, "tripped": False}),
        lt3way_mode="manual", lt3way_cooler_pct=25, ht3way_cooler_pct=95,
        strainer_fouling=15, cooler_fouling=12,
        sw_supply_temp=28, sw_discharge_temp=33, sw_flow_total=0, sw_pressure=0.1, strainer_delta_p=0,
        lt_supply_temp=52.8, lt_return_temp=62.5, lt_flow_total=360, lt_pressure=2.8,
        ht_inlet_temp=84.5, ht_outlet_temp=91.6, ht_flow_total=240, ht_pressure=3.1,
        lube_oil_temp=66.8, scavenge_air_temp=58.0,
        alarms=[
            alarm_awal("critical", "SW Pump #1",
                       "TRIP: Beban berlebih motor listrik! Aliran air laut terhenti (0 m³/h)."),
            alarm_awal("warning", "LT 3-Way Valve",
                       "Katup terjebak di mode Manual 25%! Ubah ke Auto atau buka penuh ke Central Cooler."),
        ],
    )


def tujuan_emergency(s):
    sw2_jalan = s["pumps"]["sw2"]["running"] and s["sw_flow_total"] > 250
    katup_benar = s["lt3way_mode"] == "auto" or s["lt3way_cooler_pct"] >= 70
    tidak_trip = not s["engine_tripped"]
    stabil = s["mission_hold_seconds"] >= 15
    return [
        {"id": "standby", "label": "Aktifkan Pompa SW #2 (Standby)", "completed": sw2_jalan,
         "detail": f"Aliran {s['sw_flow_total']:.0f} m³/h" if sw2_jalan else "Pompa SW #2 masih mati"},
        {"id": "valve", "label": "Pulihkan Katup 3-Arah LT (Auto / ≥ 70%)", "completed": katup_benar,
         "detail": "Mode Auto aktif" if s["lt3way_mode"] == "auto" else f"Manual {s['lt3way_cooler_pct']:.0f}%"},
        {"id": "notrip", "label": "Cegah Engine Auto-Shutdown (< 98°C)", "completed": tidak_trip,
         "detail": f"HT {s['ht_outlet_temp']:.1f}°C" if tidak_trip else "MESIN TRIP!"},
        {"id": "recover", "label": "Pulihkan daya ≥ 75% & stabilkan HT < 87°C (15 dtk)", "completed": stabil,
         "detail": f"{min(15, s['mission_hold_seconds']):.0f}/15 dtk · Beban {s['actual_load_pct']:.0f}%"},
    ]


# ---------------------------------------------------------------- Daftar skenario

SCENARIOS = {
    "sandbox": {
        "id": "sandbox",
        "title": "Mode Bebas (Sandbox)",
        "difficulty": "Eksplorasi",
        "briefing": "Kendalikan seluruh sistem pendingin kapal secara bebas. Ubah beban mesin utama & auxiliary "
                    "engine, suhu perairan, pompa, bukaan katup manual, atau uji skenario gangguan.",
        "buat_kondisi": buat_sandbox,
        "cek_tujuan": lambda s: [],
    },
    "warmup": {
        "id": "warmup",
        "title": "Misi 1: Pemanasan & Berangkat Pelabuhan",
        "difficulty": "Dasar",
        "briefing": "Mesin utama dingin (28°C) saat bersandar, sementara Auxiliary Engine memasok listrik pelabuhan "
                    "(35%). Jalankan sirkulasi HT & LT, panaskan jaket ≥ 58°C, buka katup overboard & pompa SW, "
                    "lalu berlayar ≥ 70%.",
        "buat_kondisi": buat_warmup,
        "cek_tujuan": tujuan_warmup,
    },
    "tropical": {
        "id": "tropical",
        "title": "Misi 2: Perairan Tropis & Fouling",
        "difficulty": "Menengah",
        "briefing": "Kapal melintasi perairan tropis 32°C dengan beban mesin 85% & Auxiliary Engine 65%. Filter air "
                    "laut tersumbat kotoran (68%) dan Central Cooler berkerak sehingga suhu HT mulai melonjak "
                    "mendekati batas alarm!",
        "buat_kondisi": buat_tropical,
        "cek_tujuan": tujuan_tropical,
    },
    "emergency": {
        "id": "emergency",
        "title": "Misi 3: Kegagalan Pompa Utama (Darurat)",
        "difficulty": "Tantangan",
        "briefing": "DARURAT! Pompa SW #1 mengalami trip elektrik mendadak pada beban mesin 88%, dan katup 3-arah "
                    "LT terjebak di mode Manual (25% ke cooler). Suhu jaket melesat ke 91°C! Selamatkan mesin "
                    "sebelum auto-shutdown (98°C)!",
        "buat_kondisi": buat_emergency,
        "cek_tujuan": tujuan_emergency,
    },
}

## 3. Aksi operatorPompa, katup, backwash, reset trip, dan gangguan. Semua fungsi mengubah kondisi kapal `s`.

In [ ]:
"""
Aksi operator: menyalakan pompa, backwash filter, reset trip, memasukkan gangguan, dan sebagainya.
Terjemahan dari fungsi handle... di src/App.tsx pada versi web.

Semua fungsi mengubah kondisi kapal s secara langsung.
"""


def catat(s, tingkat, sumber, pesan, acknowledged=True):
    """Tambahkan satu baris ke log alarm (maksimal 40 baris, yang terbaru di depan)."""
    s["alarms"] = ([{
        "waktu": s["elapsed_seconds"],
        "tingkat": tingkat,
        "sumber": sumber,
        "pesan": pesan,
        "acknowledged": acknowledged,
    }] + s["alarms"])[:40]


# ---------------------------------------------------------------- Pompa dan katup

def toggle_pompa(s, id_pompa):
    """Nyalakan/matikan pompa. Pompa yang sedang trip tidak bisa dinyalakan (reset dulu)."""
    pompa = s["pumps"][id_pompa]
    if pompa["tripped"]:
        return
    pompa["running"] = not pompa["running"]


def reset_trip_pompa(s, id_pompa):
    pompa = s["pumps"][id_pompa]
    pompa["tripped"] = False
    pompa["running"] = True


def toggle_katup_pompa(s, id_pompa):
    pompa = s["pumps"][id_pompa]
    pompa["valve_open"] = not pompa["valve_open"]
    if pompa["valve_open"] and pompa["valve_pct"] == 0:
        pompa["valve_pct"] = 100


def atur_katup_pompa(s, id_pompa, persen):
    """Atur bukaan katup discharge pompa (0 sampai 100)."""
    pompa = s["pumps"][id_pompa]
    pompa["valve_open"] = persen > 0
    pompa["valve_pct"] = persen


def toggle_sea_chest_valve(s):
    s["sea_chest_valve_open"] = not s["sea_chest_valve_open"]
    if s["sea_chest_valve_open"] and s["sea_chest_valve_pct"] == 0:
        s["sea_chest_valve_pct"] = 100


def atur_sea_chest_valve(s, persen):
    s["sea_chest_valve_open"] = persen > 0
    s["sea_chest_valve_pct"] = persen


def toggle_sea_chest_mode(s):
    """Ganti sea chest antara low (bawah) dan high (atas)."""
    s["sea_chest_mode"] = "high" if s["sea_chest_mode"] == "low" else "low"


def toggle_overboard(s):
    s["overboard_valve_open"] = not s["overboard_valve_open"]
    if s["overboard_valve_open"] and s["overboard_valve_pct"] == 0:
        s["overboard_valve_pct"] = 100


def atur_overboard(s, persen):
    s["overboard_valve_open"] = persen > 0
    s["overboard_valve_pct"] = persen


def mode_3way(s, loop, mode):
    """loop: 'lt' atau 'ht'. mode: 'auto' atau 'manual'."""
    s[f"{loop}3way_mode"] = mode


def atur_3way(s, loop, persen):
    """Bukaan katup 3-arah ke cooler (hanya berpengaruh kalau mode manual)."""
    s[f"{loop}3way_cooler_pct"] = persen


def toggle_pemanas_jaket(s):
    s["jacket_heater_on"] = not s["jacket_heater_on"]


# ---------------------------------------------------------------- Perawatan

def backwash_strainer(s):
    s["strainer_fouling"] = 4
    catat(s, "info", "SW Strainer", "Backwash selesai. Filter air laut kembali bersih (4%).")


def bersihkan_cooler(s):
    s["cooler_fouling"] = 3
    catat(s, "info", "Central Cooler", "Pembersihan pelat titanium selesai. Efisiensi penukar panas pulih (97%).")


def reset_trip_mesin(s):
    """Reset safety shutdown. Hanya bisa kalau suhu jaket HT sudah di bawah 92°C. Return True kalau berhasil."""
    if s["ht_outlet_temp"] >= 92:
        return False
    s["engine_tripped"] = False
    s["engine_trip_reason"] = None
    s["telegraph_pct"] = min(s["telegraph_pct"], 35)
    catat(s, "info", "Main Engine", "Safety shutdown direset. Silakan naikkan beban mesin secara bertahap.")
    return True


def acknowledge_semua(s):
    for a in s["alarms"]:
        a["acknowledged"] = True


# ---------------------------------------------------------------- Gangguan untuk uji coba

def injeksi_gangguan(s, jenis):
    """jenis: 'sw1_trip', 'strainer_clog', 'cooler_foul', atau 'lt_valve_stuck'."""
    if jenis == "sw1_trip":
        s["pumps"]["sw1"]["running"] = False
        s["pumps"]["sw1"]["tripped"] = True
        catat(s, "critical", "SW Pump #1", "GANGGUAN DIUJI: Pompa SW #1 mengalami trip elektrik!", False)
    elif jenis == "strainer_clog":
        s["strainer_fouling"] = 74
        catat(s, "warning", "SW Strainer", "GANGGUAN DIUJI: Sampah laut menyumbat Filter SW (74%).", False)
    elif jenis == "cooler_foul":
        s["cooler_fouling"] = 60
        catat(s, "warning", "Central Cooler", "GANGGUAN DIUJI: Terbentuk kerak tebal pada Central Cooler (60%).", False)
    elif jenis == "lt_valve_stuck":
        s["lt3way_mode"] = "manual"
        s["lt3way_cooler_pct"] = 20
        catat(s, "warning", "LT 3-Way Valve",
              "GANGGUAN DIUJI: Katup 3-Arah LT beralih ke Manual (20% ke cooler).", False)
    else:
        raise ValueError(f"Jenis gangguan tidak dikenal: {jenis}")

## 4. Tampilan (ringkasan dan grafik)

In [ ]:
"""
Fungsi untuk menampilkan hasil simulasi: ringkasan teks, log alarm, daftar misi, dan grafik.
"""

import matplotlib.pyplot as plt


NAMA_POMPA = {
    "sw1": "Pompa SW #1", "sw2": "Pompa SW #2",
    "lt1": "Pompa LT #1", "lt2": "Pompa LT #2",
    "ht1": "Pompa HT #1", "ht2": "Pompa HT #2",
}


def status_pompa(pompa):
    if pompa["tripped"]:
        return "TRIP"
    if not pompa["running"]:
        return "mati"
    return f"jalan ({pompa['valve_pct']:.0f}%)"


def ringkasan(s):
    """Teks ringkasan kondisi kapal saat ini."""
    judul = SCENARIOS[s["scenario_id"]]["title"]
    baris = [
        f"{judul}   |   waktu simulasi {s['elapsed_seconds']:.0f} dtk",
        "-" * 60,
        f"Mesin utama : beban {s['actual_load_pct']:.0f}%  ({s['engine_rpm']} rpm)"
        + (f"   TRIP: {s['engine_trip_reason']}" if s["engine_tripped"] else ""),
        f"Aux engine  : beban {s['aux_load_pct']:.0f}%  suhu {s['aux_engine_temp']:.1f} °C",
        f"Air laut    : {s['sea_water_temp']:.0f} °C masuk, {s['sw_discharge_temp']:.1f} °C keluar, "
        f"{s['sw_flow_total']:.0f} m³/h, {s['sw_pressure']:.1f} bar",
        f"LT          : supply {s['lt_supply_temp']:.1f} °C, return {s['lt_return_temp']:.1f} °C, "
        f"{s['lt_flow_total']:.0f} m³/h",
        f"HT (jaket)  : masuk {s['ht_inlet_temp']:.1f} °C, keluar {s['ht_outlet_temp']:.1f} °C, "
        f"{s['ht_flow_total']:.0f} m³/h",
        f"Lube oil    : {s['lube_oil_temp']:.1f} °C   Scavenge air: {s['scavenge_air_temp']:.1f} °C",
        f"Kotor       : filter SW {s['strainer_fouling']:.0f}% (ΔP {s['strainer_delta_p']:.2f} bar), "
        f"central cooler {s['cooler_fouling']:.0f}%",
        f"Katup 3-arah: LT {s['lt3way_mode']} {s['lt3way_cooler_pct']:.0f}%,  "
        f"HT {s['ht3way_mode']} {s['ht3way_cooler_pct']:.0f}%",
        "Pompa       : " + ", ".join(f"{NAMA_POMPA[k]} {status_pompa(v)}" for k, v in s["pumps"].items()),
    ]
    return "\n".join(baris)


def cetak_alarm(s, jumlah=10):
    """Cetak beberapa alarm terbaru."""
    tanda = {"info": "INFO", "warning": "WARN", "critical": "CRIT"}
    for a in s["alarms"][:jumlah]:
        print(f"[{a['waktu']:6.1f} dtk] {tanda[a['tingkat']]}  {a['sumber']}: {a['pesan']}")


def cetak_tujuan(s):
    """Cetak daftar tujuan misi dan apakah sudah tercapai."""
    daftar = SCENARIOS[s["scenario_id"]]["cek_tujuan"](s)
    for t in daftar:
        centang = "[x]" if t["completed"] else "[ ]"
        print(f"{centang} {t['label']}  ({t['detail']})")
    if s["mission_completed"]:
        print("MISI SELESAI")
    elif s["mission_failed"]:
        print("MISI GAGAL (mesin trip)")


def grafik(s, simpan_ke=None, tampil=True):
    """Gambar tiga grafik: suhu, beban mesin, dan aliran air laut. Return objek figure."""
    r = s["history"]
    fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    if not r:
        a1.text(0.5, 0.5, "Belum ada data. Jalankan simulasi dulu.", ha="center", transform=a1.transAxes)
    else:
        t = [p["time"] for p in r]
        a1.plot(t, [p["ht_outlet"] for p in r], label="HT keluar jaket", linewidth=2)
        a1.plot(t, [p["lt_supply"] for p in r], label="LT supply")
        a1.plot(t, [p["lube_oil"] for p in r], label="Lube oil")
        a1.plot(t, [p["scavenge_air"] for p in r], label="Scavenge air")
        a1.plot(t, [p["aux_engine"] for p in r], label="Aux engine")
        a1.axhline(98, color="red", linestyle="--", linewidth=1)
        a1.text(t[-1], 98.8, "batas trip HT 98 °C", color="red", fontsize=8, ha="right")
        a1.set_ylabel("Suhu (°C)")
        a1.set_ylim(top=112)
        a1.legend(loc="upper left", ncol=3, fontsize=8)
        a2.plot(t, [p["engine_load"] for p in r], color="tab:orange")
        a2.set_ylabel("Beban mesin (%)")
        a3.plot(t, [p["sw_flow"] for p in r], color="tab:blue")
        a3.set_ylabel("Aliran SW (m³/h)")
    a3.set_xlabel("Waktu simulasi (detik)")
    for a in (a1, a2, a3):
        a.grid(alpha=0.3)
    fig.suptitle(SCENARIOS[s["scenario_id"]]["title"])
    fig.tight_layout()
    if simpan_ke:
        fig.savefig(simpan_ke, dpi=130)
    if tampil:
        plt.show()
    return fig

## 5. Contoh tanpa panel: Misi 1 diselesaikan lewat kodeCara paling sederhana memakai simulator: buat kondisi awal, jalankan langkah demi langkah, ubah pengaturan lewat kode.

In [ ]:
DT = 0.25   # tiap langkah = 0,25 detik waktu simulasi


def jalankan(s, detik, tiap_langkah=None):
    """Jalankan simulasi sebanyak 'detik' detik. tiap_langkah(s, t) dipanggil sebelum setiap langkah."""
    for i in range(1, int(detik / DT) + 1):
        if tiap_langkah:
            tiap_langkah(s, i * DT)
        langkah_simulasi(s, DT)
        if s["mission_completed"] or s["engine_tripped"]:
            break
    return s


s = SCENARIOS["warmup"]["buat_kondisi"]()

# Langkah awal: nyalakan sirkulasi, buka overboard, hidupkan pemanas jaket
for pompa in ("sw1", "lt1", "ht1"):
    toggle_pompa(s, pompa)
toggle_overboard(s)
toggle_pemanas_jaket(s)
sudah_panas = False


def tiap(s, t):
    global sudah_panas
    # Setelah jaket mencapai 60 °C: matikan pemanas, naikkan beban, buka penuh katup 3-arah
    if not sudah_panas and s["ht_outlet_temp"] >= 60:
        sudah_panas = True
        s["jacket_heater_on"] = False
        s["telegraph_pct"] = 72
        for loop in ("lt", "ht"):
            mode_3way(s, loop, "manual")
            atur_3way(s, loop, 100)


jalankan(s, 300, tiap)
print(ringkasan(s))
print()
cetak_tujuan(s)
grafik(s)

## 6. Panel interaktifPilih skenario, tekan **Play** di bagian kontrol waktu, lalu atur beban mesin, pompa, dan katup seperti di aplikasi web. Tombol **+10 detik** menjalankan simulasi lebih cepat tanpa menunggu.

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output

panel = {"s": SCENARIOS["sandbox"]["buat_kondisi"](), "tick": 0}
sedang_sinkron = False   # True saat kode (bukan pengguna) yang mengubah nilai slider

keluaran_teks = w.Output()
keluaran_grafik = w.Output()
teks_misi = w.HTML()


def gambar_teks():
    s = panel["s"]
    with keluaran_teks:
        clear_output(wait=True)
        print(ringkasan(s))
        print()
        cetak_tujuan(s)
        print()
        cetak_alarm(s, 5)


def gambar_grafik():
    with keluaran_grafik:
        clear_output(wait=True)
        fig = grafik(panel["s"], tampil=False)
        display(fig)
        plt.close(fig)


# ---------- Pilih skenario
pilih_skenario = w.Dropdown(options=[(v["title"], k) for k, v in SCENARIOS.items()], description="Skenario")
tombol_mulai = w.Button(description="Mulai / reset skenario", button_style="primary")

# ---------- Kontrol waktu
play = w.Play(value=0, min=0, max=10**7, step=1, interval=500)
kecepatan = w.Dropdown(options=[("1x", 1), ("2x", 2), ("4x", 4)], value=1, description="Kecepatan")
tombol_10 = w.Button(description="+10 detik")

# ---------- Slider utama
sl_beban = w.IntSlider(value=65, min=0, max=100, description="Beban mesin %")
sl_aux = w.IntSlider(value=45, min=0, max=100, description="Beban aux %")
sl_laut = w.IntSlider(value=24, min=2, max=34, description="Suhu laut °C")
sl_overboard = w.IntSlider(value=100, min=0, max=100, description="Overboard %")
sl_seachest = w.IntSlider(value=100, min=0, max=100, description="Sea chest %")

# ---------- Katup 3-arah
mode_lt = w.ToggleButtons(options=["auto", "manual"], description="Katup LT")
sl_lt = w.IntSlider(value=50, min=0, max=100, description="LT ke cooler %")
mode_ht = w.ToggleButtons(options=["auto", "manual"], description="Katup HT")
sl_ht = w.IntSlider(value=50, min=0, max=100, description="HT ke cooler %")

# ---------- Katup discharge tiap pompa
pilih_pompa = w.Dropdown(options=[(NAMA_POMPA[k], k) for k in NAMA_POMPA], description="Katup pompa")
sl_katup_pompa = w.IntSlider(value=100, min=0, max=100, description="Bukaan %")

# ---------- Tombol pompa dan perawatan
tombol_pompa = {k: w.Button(description=NAMA_POMPA[k], layout=w.Layout(width="150px")) for k in NAMA_POMPA}
tombol_pemanas = w.Button(description="Pemanas jaket", layout=w.Layout(width="150px"))
tombol_backwash = w.Button(description="Backwash filter SW")
tombol_cooler = w.Button(description="Bersihkan cooler")
tombol_reset_mesin = w.Button(description="Reset trip mesin", button_style="warning")
tombol_acknowledge = w.Button(description="Acknowledge alarm")

pilih_gangguan = w.Dropdown(
    options=[("Pompa SW #1 trip", "sw1_trip"), ("Filter SW tersumbat", "strainer_clog"),
             ("Central cooler berkerak", "cooler_foul"), ("Katup 3-arah LT macet", "lt_valve_stuck")],
    description="Gangguan")
tombol_gangguan = w.Button(description="Beri gangguan", button_style="danger")


def perbarui_tombol():
    """Samakan tulisan dan warna tombol dengan kondisi kapal."""
    s = panel["s"]
    for k, b in tombol_pompa.items():
        p = s["pumps"][k]
        if p["tripped"]:
            b.description, b.button_style = f"{NAMA_POMPA[k]}: TRIP (reset)", "danger"
        elif p["running"]:
            b.description, b.button_style = f"{NAMA_POMPA[k]}: jalan", "success"
        else:
            b.description, b.button_style = f"{NAMA_POMPA[k]}: mati", ""
    tombol_pemanas.description = "Pemanas jaket: " + ("nyala" if s["jacket_heater_on"] else "mati")
    tombol_pemanas.button_style = "success" if s["jacket_heater_on"] else ""


def segarkan():
    perbarui_tombol()
    gambar_teks()


# ---------- Isi slider dari kondisi kapal (dipakai saat skenario baru dimulai)
def sinkron_dari_kondisi():
    global sedang_sinkron
    s = panel["s"]
    sedang_sinkron = True
    sl_beban.value = int(s["telegraph_pct"])
    sl_aux.value = int(s["aux_load_pct"])
    sl_laut.value = int(s["sea_water_temp"])
    sl_overboard.value = int(s["overboard_valve_pct"]) if s["overboard_valve_open"] else 0
    sl_seachest.value = int(s["sea_chest_valve_pct"]) if s["sea_chest_valve_open"] else 0
    mode_lt.value, mode_ht.value = s["lt3way_mode"], s["ht3way_mode"]
    sl_lt.value, sl_ht.value = int(s["lt3way_cooler_pct"]), int(s["ht3way_cooler_pct"])
    sl_katup_pompa.value = int(s["pumps"][pilih_pompa.value]["valve_pct"])
    kecepatan.value = s["speed_multiplier"]
    sedang_sinkron = False
    teks_misi.value = (f"<b>{SCENARIOS[s['scenario_id']]['title']}</b> "
                       f"({SCENARIOS[s['scenario_id']]['difficulty']})<br>{SCENARIOS[s['scenario_id']]['briefing']}")


# ---------- Hubungkan slider ke kondisi kapal
def hubungkan(slider, fungsi):
    def ubah(perubahan):
        if not sedang_sinkron:
            fungsi(panel["s"], perubahan["new"])
    slider.observe(ubah, names="value")


def set_nilai(kunci):
    def f(s, nilai):
        s[kunci] = nilai
    return f


hubungkan(sl_beban, set_nilai("telegraph_pct"))
hubungkan(sl_aux, set_nilai("aux_load_pct"))
hubungkan(sl_laut, set_nilai("sea_water_temp"))
hubungkan(sl_overboard, atur_overboard)
hubungkan(sl_seachest, atur_sea_chest_valve)
hubungkan(sl_lt, lambda s, v: atur_3way(s, "lt", v))
hubungkan(sl_ht, lambda s, v: atur_3way(s, "ht", v))
hubungkan(sl_katup_pompa, lambda s, v: atur_katup_pompa(s, pilih_pompa.value, v))


def ganti_mode(loop, mode_widget, slider):
    def ubah(perubahan):
        global sedang_sinkron
        if sedang_sinkron:
            return
        s = panel["s"]
        mode_3way(s, loop, perubahan["new"])
        if perubahan["new"] == "manual":   # mulai dari bukaan yang sedang berjalan
            sedang_sinkron = True
            slider.value = int(s[f"{loop}3way_cooler_pct"])
            sedang_sinkron = False
        segarkan()
    mode_widget.observe(ubah, names="value")


ganti_mode("lt", mode_lt, sl_lt)
ganti_mode("ht", mode_ht, sl_ht)


def ganti_pompa(perubahan):
    global sedang_sinkron
    sedang_sinkron = True
    sl_katup_pompa.value = int(panel["s"]["pumps"][perubahan["new"]]["valve_pct"])
    sedang_sinkron = False


pilih_pompa.observe(ganti_pompa, names="value")
kecepatan.observe(lambda c: panel["s"].update(speed_multiplier=c["new"]), names="value")


# ---------- Tombol
def klik_pompa(k):
    def f(_):
        s = panel["s"]
        if s["pumps"][k]["tripped"]:
            reset_trip_pompa(s, k)
        else:
            toggle_pompa(s, k)
        segarkan()
    return f


for k, b in tombol_pompa.items():
    b.on_click(klik_pompa(k))


def klik(fungsi):
    def f(_):
        fungsi(panel["s"])
        segarkan()
    return f


tombol_pemanas.on_click(klik(toggle_pemanas_jaket))
tombol_backwash.on_click(klik(backwash_strainer))
tombol_cooler.on_click(klik(bersihkan_cooler))
tombol_reset_mesin.on_click(klik(reset_trip_mesin))
tombol_acknowledge.on_click(klik(acknowledge_semua))
tombol_gangguan.on_click(lambda _: (injeksi_gangguan(panel["s"], pilih_gangguan.value), segarkan()))


def mulai_skenario(_):
    panel["s"] = SCENARIOS[pilih_skenario.value]["buat_kondisi"]()
    panel["tick"] = 0
    sinkron_dari_kondisi()
    segarkan()
    gambar_grafik()


tombol_mulai.on_click(mulai_skenario)


# ---------- Jalannya waktu
def tiap_tick(perubahan):
    s = panel["s"]
    for _ in range(2):          # 2 langkah x 0,25 dtk = 0,5 dtk, sama dengan interval Play
        langkah_simulasi(s, DT)
    panel["tick"] += 1
    segarkan()
    if panel["tick"] % 4 == 0:  # grafik digambar ulang tiap 2 detik supaya tidak berat
        gambar_grafik()


play.observe(tiap_tick, names="value")


def maju_10(_):
    s = panel["s"]
    for _ in range(int(10 / DT)):
        langkah_simulasi(s, DT)
    segarkan()
    gambar_grafik()


tombol_10.on_click(maju_10)

# ---------- Susun tampilan
tampilan = w.VBox([
    w.HBox([pilih_skenario, tombol_mulai]),
    teks_misi,
    w.HBox([play, kecepatan, tombol_10]),
    w.HTML("<b>Beban dan lingkungan</b>"),
    w.VBox([sl_beban, sl_aux, sl_laut]),
    w.HTML("<b>Pompa</b>"),
    w.HBox([tombol_pompa["sw1"], tombol_pompa["sw2"], tombol_pompa["lt1"]]),
    w.HBox([tombol_pompa["lt2"], tombol_pompa["ht1"], tombol_pompa["ht2"]]),
    w.HBox([pilih_pompa, sl_katup_pompa]),
    w.HTML("<b>Katup</b>"),
    w.VBox([sl_overboard, sl_seachest, mode_lt, sl_lt, mode_ht, sl_ht]),
    w.HTML("<b>Perawatan dan gangguan</b>"),
    w.HBox([tombol_pemanas, tombol_backwash, tombol_cooler]),
    w.HBox([tombol_reset_mesin, tombol_acknowledge]),
    w.HBox([pilih_gangguan, tombol_gangguan]),
    keluaran_teks,
    keluaran_grafik,
])

sinkron_dari_kondisi()
perbarui_tombol()
display(tampilan)
gambar_teks()
gambar_grafik()